In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import numpy as np
from tqdm import tqdm  
import time

from sklearn.preprocessing import StandardScaler
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import os
import numpy as np
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset

import os
import gc
import shutil

# 检查 GPU 可用性
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 1.加载数据集（初始）

In [ ]:
class StockDataset_by_year(Dataset):
    def __init__(self, base_dir, start_year, end_year, task='regression'):
        self.base_dir = base_dir
        self.years = list(range(start_year, end_year + 1))
        self.task = task
        self.data = []
        self.labels = []
        
        # 加载数据并使用进度条
        for year in tqdm(self.years, desc="Loading Data by Year"):
            data_file = os.path.join(base_dir, f"{year}_df_data.feather")
            label_file = os.path.join(base_dir, f"{year}_real_RET.feather")
            
            # 读取数据和标签
            X = pd.read_feather(data_file)
            y = pd.read_feather(label_file)
            
            # 删除 'permno' 和 'DATE' 列，只保留剩余列作为特征
            X = X.drop(columns=['permno', 'DATE'])
            
            # 如果任务是分类，则将y的值转化为0或1
            if self.task == 'classification':
                y = (y > 0).astype(int)
            
            # 转换为numpy数组并存储
            self.data.append(X.values)
            self.labels.append(y.values)
        
        # 将数据和标签转换为单个数组
        self.data = torch.tensor(np.vstack(self.data), dtype=torch.float32)
        self.labels = torch.tensor(np.vstack(self.labels), dtype=torch.float32)
        
        # 打印数据集信息
        print(f"Data loaded from {start_year} to {end_year}.")
        print(f"Total samples: {len(self.labels)}")
        print(f"Feature dimension: {self.data.shape[1]}")
        print(f"Task: {self.task}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]
    


# 构建数据集和数据加载器的函数
def create_dataloaders(base_dir, start_year_train, start_year_test, end_year_test, task='regression', batch_size=128):
    # 构建训练集
    print("Constructing training dataset...")
    train_dataset = StockDataset_by_year(base_dir, start_year_train, start_year_test - 1, task)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # 构建测试集
    print("Constructing testing dataset...")
    test_dataset = StockDataset_by_year(base_dir, start_year_test, end_year_test, task)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True) 

    return train_loader, test_loader

In [ ]:
def calculate_r2_oos(real, predict):
    real = np.array(real)
    predict = np.array(predict)
    assert real.shape == predict.shape 
    valid_mask = ~np.isnan(real)
    numerator = np.sum((real[valid_mask] - predict[valid_mask]) ** 2)
    denominator = np.sum(real[valid_mask] ** 2)
    R2_oos = 1 - (numerator / denominator)
    return R2_oos

## 2.模型集成训练

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
import numpy as np
import random
import time
import os
from torch.utils.data import DataLoader
from tqdm import tqdm

# Set random seed for reproducibility
def set_random_seed(run_idx):
    torch.manual_seed(run_idx)  # Set CPU random seed
    torch.cuda.manual_seed_all(run_idx)  # Set GPU random seed
    np.random.seed(run_idx)  # Set numpy random seed
    random.seed(run_idx)  # Set Python random seed

# DataLoader worker init function for shuffling data with different seed
def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)

# Define the NN3 model class
class NN3(nn.Module):
    def __init__(self, input_dim, hidden_neurons1=64, hidden_neurons2=64, hidden_neurons3=32, dropout_rate1=0.3, dropout_rate2=0.4, dropout_rate3=0.5):
        super(NN3, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_neurons1)
        self.bn1 = nn.BatchNorm1d(hidden_neurons1)
        self.fc2 = nn.Linear(hidden_neurons1, hidden_neurons2)
        self.bn2 = nn.BatchNorm1d(hidden_neurons2)
        self.fc3 = nn.Linear(hidden_neurons2, hidden_neurons3)
        self.bn3 = nn.BatchNorm1d(hidden_neurons3)
        self.out = nn.Linear(hidden_neurons3, 1)
        self.dropout1 = nn.Dropout(dropout_rate1)
        self.dropout2 = nn.Dropout(dropout_rate2)
        self.dropout3 = nn.Dropout(dropout_rate3)
    
    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.dropout3(x)
        return self.out(x)

# Training function for a single model
def train_single_model(model, train_loader, test_loader, input_dim, num_epochs=50, lr=0.001, patience=10, save_base_path=None, use_scheduler=False):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)  # Added weight decay for L2 regularization
    
    # Optionally add a learning rate scheduler
    scheduler = None
    if use_scheduler:
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

    best_r2 = float('-inf')
    best_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    results = {'train_loss': [], 'test_loss': [], 'test_r2': [], 'learning_rate': []}

    # Create model folder for each model
    model_folder = os.path.join(save_base_path, f"{model.__class__.__name__}_{str(time.time()).split('.')[0]}")
    if not os.path.exists(model_folder):
        os.makedirs(model_folder)

    total_start_time = time.time()

    # Training loop
    epoch_iterator = tqdm(range(num_epochs), desc='Training Epochs', unit='epoch')
    for epoch in epoch_iterator:
        model.train()
        train_loss = 0.0
        train_loader_iter = tqdm(train_loader, desc=f"Epoch {epoch+1} Training", leave=False, unit='batch')
        for X_batch, y_batch in train_loader_iter:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        
        train_loss /= len(train_loader.dataset)
        results['train_loss'].append(train_loss)

        model.eval()
        test_loss = 0.0
        y_test_pred_list = []
        y_test_list = []
        test_loader_iter = tqdm(test_loader, desc=f"Epoch {epoch+1} Testing", leave=False, unit='batch')
        with torch.no_grad():
            for X_batch, y_batch in test_loader_iter:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                test_loss += loss.item() * X_batch.size(0)
                y_test_pred_list.append(outputs.cpu().numpy())
                y_test_list.append(y_batch.cpu().numpy())
        
        test_loss /= len(test_loader.dataset)
        y_test_pred = np.concatenate(y_test_pred_list).ravel()
        y_test = np.concatenate(y_test_list).ravel()
        test_r2 = calculate_r2_oos(y_test, y_test_pred)
        
        results['test_loss'].append(test_loss)
        results['test_r2'].append(test_r2)

        # Scheduler step
        if scheduler:
            scheduler.step(test_loss)

        # Early stopping check based on test_loss and test_r2
        if test_loss < best_loss and test_r2 > best_r2:
            best_loss = test_loss
            best_r2 = test_r2
            patience_counter = 0
            best_model_state = model.state_dict()
        else:
            patience_counter += 1
            if patience_counter >= patience:
                break

    total_training_time = time.time() - total_start_time

    # Save best model
    if best_model_state:
        model.load_state_dict(best_model_state)
        model_save_path = os.path.join(model_folder, f"best_model.pth")
        torch.save(model.state_dict(), model_save_path)

        # Save training history
        history_save_path = os.path.join(model_folder, "training_history.txt")
        with open(history_save_path, 'w') as f:
            f.write("Epoch\tTrain Loss\tTest Loss\tTest R²\tLR\n")
            for epoch in range(len(results['train_loss'])):
                f.write(f"{epoch+1}\t{results['train_loss'][epoch]:.6f}\t{results['test_loss'][epoch]:.6f}\t{results['test_r2'][epoch]:.6f}\n")

        return model_save_path, results['test_r2'][-1], total_training_time
    else:
        return None, results['test_r2'][-1], total_training_time

# Training multiple models
def train_multiple_models(model_class, train_loader, test_loader, input_dim, num_models=5, num_epochs=50, lr=0.001, patience=10, save_base_path=None):
    model_paths = []
    final_r2_scores = []
    total_times = []

    for model_num in range(num_models):
        print(f"Training model {model_num+1} of {num_models}")
        
        # Set random seed for each model
        set_random_seed(model_num)
        
        # Create a new instance of the model
        model = model_class(input_dim)
        
        model_path, final_r2, total_duration = train_single_model(
            model, train_loader, test_loader, input_dim, num_epochs, lr, patience, save_base_path
        )
        
        model_paths.append(model_path)
        final_r2_scores.append(final_r2)
        total_times.append(total_duration)

        # Clear GPU memory after each model training
        torch.cuda.empty_cache()

    # Optionally, save the ensemble performance summary
    ensemble_summary = {
        'model_paths': model_paths,
        'final_r2_scores': final_r2_scores,
        'total_times': total_times
    }
    
    summary_path = os.path.join(save_base_path, "ensemble_summary.npy")
    np.save(summary_path, ensemble_summary)
    
    return ensemble_summary

In [ ]:
# Example usage:
data_dir = 'data/processed/yearly'
start_year_train = 1957
start_year_test = 1987
end_year_test = 2016
train_loader, test_loader = create_dataloaders(data_dir, start_year_train, start_year_test, end_year_test, task='regression')

input_dim = train_loader.dataset[0][0].shape[0]
base_save_path = 'outputs/models'

In [ ]:
# 训练多个模型
ensemble_summary = train_multiple_models(
    NN3, train_loader, test_loader, input_dim, num_models=5, num_epochs=15, lr=0.001, patience=5, save_base_path=base_save_path
)

# 打印结果
print(f"Ensemble training complete. Model paths: {ensemble_summary['model_paths']}")
print(f"Final R² scores: {ensemble_summary['final_r2_scores']}")
print(f"Total training times: {ensemble_summary['total_times']}")

# 清理内存
gc.collect()

## 3.加载模型并验证(新dataset加载)

In [ ]:
class StockDataset_by_year(Dataset):
    # task任务类型，默认为回归
    def __init__(self, base_dir, start_year, end_year, task='regression'):
        self.base_dir = base_dir
        self.years = list(range(start_year, end_year + 1))
        self.task = task

        self.data_full = []
        self.features_list = []
        self.labels_list = []

        # 加载数据并使用进度条
        for year in tqdm(self.years, desc="Loading Data by Year"):
            data_file = os.path.join(base_dir, f"{year}_df_data.feather")
            label_file = os.path.join(base_dir, f"{year}_real_RET.feather")

            # 读取数据和标签
            X = pd.read_feather(data_file)
            y = pd.read_feather(label_file)
            print(f"Loaded data for year {year}:")

            # 如果任务是分类，则将y的值转化为0或1
            if self.task == 'classification':
                y['real_RET'] = (y['real_RET'] > 0).astype(float)

            combined_df = pd.concat([X, y['real_RET']], axis=1)

            reserved_cols = ['permno', 'DATE', 'mvel1', 'real_RET']
            feature_cols = [col for col in X.columns if col not in ['permno', 'DATE']]

            # 避免重复列
            all_needed_cols = reserved_cols + [col for col in feature_cols if col not in ['permno', 'DATE']]

            # 构造完整数据
            self.data_full.append(combined_df[all_needed_cols])

            self.features_list.append(combined_df[feature_cols].values)
            self.labels_list.append(combined_df['real_RET'].values)

        self.data_full = pd.concat(self.data_full, ignore_index=True)
        features_all = np.vstack(self.features_list)
        labels_all = np.hstack(self.labels_list)

        self.features = torch.tensor(features_all, dtype=torch.float32)
        self.labels = torch.tensor(labels_all, dtype=torch.float32)

        print(f"Data loaded from {start_year} to {end_year}.")
        print(f"Total samples: {len(self.labels)}")
        print(f"Feature dimension: {self.features.shape[1]}")
        print(f"Task: {self.task}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        X = self.features[idx]
        y = self.labels[idx]
        return X, y, idx
    


# 构建数据集和数据加载器的函数
def create_dataloaders(base_dir, start_year_train, start_year_test, end_year_test, task='regression', batch_size=32):
    # 构建训练集
    print("Constructing training dataset...")
    train_dataset = StockDataset_by_year(base_dir, start_year_train, start_year_test - 1, task)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # 构建测试集
    print("Constructing testing dataset...")
    test_dataset = StockDataset_by_year(base_dir, start_year_test, end_year_test, task)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, test_loader




class ModelRunnerByYear:
    """
    一个可复用的类，使用以加载的模型对验证集进行预测，将预测结果输出为CSV
    可将任意PyTorch模型实例传入run()方法进行训练和验证集预测。
    """

    def __init__(self,
                 base_dir,
                 val_start_year=1987,
                 val_end_year=2016,
                 output_csv="predictions_only.csv",
                 batch_size=64,
                 task='regression'):
        self.base_dir = base_dir
        self.val_start_year = val_start_year
        self.val_end_year = val_end_year
        self.output_csv = output_csv
        self.batch_size = batch_size
        self.task = task

        print('Loading validation dataset...')
        self.val_dataset = StockDataset_by_year(self.base_dir, self.val_start_year, self.val_end_year, task=self.task)
        self.val_dataloader = DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=True)

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f'Prediction device:{self.device}')

    def run(self, model):
        model.to(self.device)
        model.eval()

        predictions = []
        indices = []
        with torch.no_grad():
            for X_batch, _, idx_batch in tqdm(self.val_dataloader, desc="Valid"):
                X_batch = X_batch.to(self.device)

                
                preds = model(X_batch).squeeze().cpu().numpy()
                predictions.append(preds)
                indices.extend(idx_batch.cpu().numpy())

        predictions = np.concatenate(predictions, axis=0)

        val_data_full = self.val_dataset.data_full.iloc[indices].copy()

        val_data_full.reset_index(drop=True, inplace=True)

        val_data_full['up_prob'] = predictions
        # 去除重复列
        val_data_full = val_data_full.loc[:, ~val_data_full.columns.duplicated()].copy()

        final_df = val_data_full[[ 'permno', 'DATE', 'up_prob', 'mvel1','real_RET']].copy()
        final_df.rename(columns={
            'permno': 'StockID',
            'DATE': 'ending_date',
            'mvel1': 'MarketCap',
            'real_RET': 'ret_val'
        }, inplace=True)
        
        # 确保列顺序
        final_df = final_df[['StockID', 'ending_date', 'up_prob', 'MarketCap', 'ret_val']]

        final_df.to_csv(self.output_csv, index=False)
        print(f"预测结果已保存至 {self.output_csv}，格式如下：")
        print(final_df.head())

In [ ]:
class NN3(nn.Module):
    def __init__(self, input_dim, hidden_neurons1=128, hidden_neurons2=64, hidden_neurons3=32, dropout_rate1=0.3, dropout_rate2=0.4, dropout_rate3=0.5):
        super(NN3, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_neurons1)
        self.bn1 = nn.BatchNorm1d(hidden_neurons1)
        self.fc2 = nn.Linear(hidden_neurons1, hidden_neurons2)
        self.bn2 = nn.BatchNorm1d(hidden_neurons2)
        self.fc3 = nn.Linear(hidden_neurons2, hidden_neurons3)
        self.bn3 = nn.BatchNorm1d(hidden_neurons3)
        self.out = nn.Linear(hidden_neurons3, 1)
        self.dropout1 = nn.Dropout(dropout_rate1)
        self.dropout2 = nn.Dropout(dropout_rate2)
        self.dropout3 = nn.Dropout(dropout_rate3)
    
    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = self.dropout1(x)
        x = torch.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.dropout3(x)
        return self.out(x)


if __name__ == "__main__":
    model = NN3(input_dim)

    # 加载事先训练好的最佳模型权重
    model.load_state_dict(torch.load('data/resources/best_model.pth'))
    

    predictor = ModelRunnerByYear(
        base_dir='data/processed/yearly',
        val_start_year=1987,
        val_end_year=2016,
        output_csv="NN3_best_predictions2.csv",
        batch_size=64,
        task='regression'
    )

    predictor.run(model)